# Batch 03 — Precision Ladder + CUDA Streams Overlap

**Experiments:**
- **3A**: TRT INT8 engine with entropy calibration → latency + precision table
- **3B**: CUDA Streams overlapped pipeline (async H→D + inference) → throughput gain
- **3C**: Precision comparison table: FP32 / FP16 / INT8 across latency, model size, GPU memory

**Runtime:** ~45–60 min total (INT8 calibration dominates)

**Prerequisites:** Run Batch 02 notebook first to have `models/yolo11n.onnx`, `models/yolo11n_fp32.engine`, `models/yolo11n_fp16.engine`, and `data/clip.mp4` ready.

> ⚠️ **Colab session note:** Engines are GPU-specific. If your session restarts or you get a different GPU, rebuild all engines. Save engines + calibration cache to Drive to skip re-calibration.

## 0. Setup

In [ ]:
import subprocess, os

# Verify GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                         '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip())

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')

import tensorrt as trt
print(f'TensorRT: {trt.__version__}')

In [ ]:
# Clone repo (skip if already present)
if not os.path.exists('/content/Inference_pipeline_eval'):
    !git clone https://github.com/YOUR_USERNAME/Inference_pipeline_eval.git /content/Inference_pipeline_eval

%cd /content/Inference_pipeline_eval
!pip install ultralytics onnxruntime-gpu --quiet

In [ ]:
# ── Google Drive (optional but strongly recommended) ──────────────────────
# Mount Drive to persist engines, calibration cache, and results across sessions.
# Skip this cell if you prefer to rebuild everything each session.

DRIVE_ROOT = '/content/drive/MyDrive/inference_pipeline'
USE_DRIVE  = True   # set False to skip

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    os.makedirs(f'{DRIVE_ROOT}/models',  exist_ok=True)
    os.makedirs(f'{DRIVE_ROOT}/results', exist_ok=True)
    os.makedirs(f'{DRIVE_ROOT}/data',    exist_ok=True)
    print(f'Drive mounted. Root: {DRIVE_ROOT}')
else:
    print('Drive not used — all files are session-local.')

In [ ]:
# ── Restore engines from Drive (skip calibration if cache exists) ─────────
import shutil

!mkdir -p models results data

if USE_DRIVE:
    for fname in ['yolo11n.onnx', 'yolo11n_fp32.engine', 'yolo11n_fp16.engine',
                  'yolo11n_int8.engine', 'calibration.cache']:
        src = f'{DRIVE_ROOT}/models/{fname}'
        dst = f'models/{fname}'
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copy(src, dst)
            print(f'Restored from Drive: {fname}')
        elif os.path.exists(dst):
            print(f'Already present   : {fname}')
        else:
            print(f'Not on Drive yet  : {fname}  (will build/generate below)')

    # Restore video
    if not os.path.exists('data/clip.mp4') and os.path.exists(f'{DRIVE_ROOT}/data/clip.mp4'):
        shutil.copy(f'{DRIVE_ROOT}/data/clip.mp4', 'data/clip.mp4')
        print('Restored: data/clip.mp4')

    # Restore calibration frames if present
    cal_src = f'{DRIVE_ROOT}/data/calibration_frames'
    cal_dst = 'data/calibration_frames'
    if os.path.exists(cal_src) and not os.path.exists(cal_dst):
        shutil.copytree(cal_src, cal_dst)
        n = len(os.listdir(cal_dst))
        print(f'Restored: calibration_frames/ ({n} images)')

## 3A-i. Upload Video (if not restored from Drive)

In [ ]:
# Run this cell only if data/clip.mp4 is not present
if not os.path.exists('data/clip.mp4'):
    from google.colab import files
    print('Upload your dashcam clip (clip.mp4):')
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    os.rename(fname, 'data/clip.mp4')
    if USE_DRIVE:
        shutil.copy('data/clip.mp4', f'{DRIVE_ROOT}/data/clip.mp4')
    print('Video ready.')
else:
    print('data/clip.mp4 already present.')

## 3A-ii. Export ONNX + Build FP32 / FP16 Engines (if not restored)

In [ ]:
if not os.path.exists('models/yolo11n.onnx'):
    !python3 scripts/export_onnx.py --model yolo11n.pt --output models/yolo11n.onnx
    if USE_DRIVE:
        shutil.copy('models/yolo11n.onnx', f'{DRIVE_ROOT}/models/yolo11n.onnx')
else:
    print('yolo11n.onnx already present.')

In [ ]:
need_fp32 = not os.path.exists('models/yolo11n_fp32.engine')
need_fp16 = not os.path.exists('models/yolo11n_fp16.engine')

if need_fp32 or need_fp16:
    flags = ('--fp32 ' if need_fp32 else '') + ('--fp16' if need_fp16 else '')
    !python3 scripts/build_tensorrt_engine.py --onnx models/yolo11n.onnx \
        --output-dir models/ {flags} --workspace-gb 2
    for fname in ['yolo11n_fp32.engine', 'yolo11n_fp16.engine']:
        if USE_DRIVE and os.path.exists(f'models/{fname}'):
            shutil.copy(f'models/{fname}', f'{DRIVE_ROOT}/models/{fname}')
else:
    print('FP32 and FP16 engines already present.')

## 3A-iii. Extract Calibration Frames

In [ ]:
cal_dir = 'data/calibration_frames'
n_existing = len(os.listdir(cal_dir)) if os.path.exists(cal_dir) else 0

if n_existing < 500:
    !python3 scripts/extract_calibration_frames.py \
        --video  data/clip.mp4 \
        --output data/calibration_frames \
        --frames 500
    # Save to Drive so we never redo this
    if USE_DRIVE:
        cal_drive = f'{DRIVE_ROOT}/data/calibration_frames'
        if not os.path.exists(cal_drive):
            shutil.copytree(cal_dir, cal_drive)
            print(f'Calibration frames saved to Drive: {cal_drive}')
else:
    print(f'Calibration frames already present ({n_existing} images).')

## 3A-iv. Build INT8 Engine

In [ ]:
# If calibration.cache exists (from Drive), engine rebuild skips re-calibration (~2 min)
# If cache missing, full calibration runs (~10–15 min)

if not os.path.exists('models/yolo11n_int8.engine'):
    !python3 scripts/build_tensorrt_engine_int8.py \
        --onnx            models/yolo11n.onnx \
        --calibration-dir data/calibration_frames \
        --cache-file      models/calibration.cache \
        --output-dir      models/ \
        --workspace-gb    2
    # Save to Drive
    if USE_DRIVE:
        for fname in ['yolo11n_int8.engine', 'calibration.cache']:
            if os.path.exists(f'models/{fname}'):
                shutil.copy(f'models/{fname}', f'{DRIVE_ROOT}/models/{fname}')
        print('INT8 engine + cache saved to Drive.')
else:
    print('INT8 engine already present.')

## 3A-v. Run INT8 Benchmark

In [ ]:
!python3 scripts/run_tensorrt_video.py \
    --video   data/clip.mp4 \
    --engine  models/yolo11n_int8.engine \
    --results results/b03_trt_int8_raw_timings.csv \
    --warmup  20

In [ ]:
# Aggregate per-frame timings into benchmark summary
!python3 scripts/benchmark.py \
    --runtime trt-int8 \
    --input   results/b03_trt_int8_raw_timings.csv \
    --output  results/b03_trt_int8_benchmark.csv \
    --warmup-frames 20

## 3B. CUDA Streams Overlapped Pipeline

In [ ]:
# Run CUDA streams pipeline (uses FP16 engine — fastest inference = most overlap benefit)
!python3 scripts/run_cuda_streams_video.py \
    --video   data/clip.mp4 \
    --engine  models/yolo11n_fp16.engine \
    --results results/b03_cuda_streams_raw_timings.csv \
    --warmup  20

In [ ]:
!python3 scripts/benchmark.py \
    --runtime cuda-streams \
    --input   results/b03_cuda_streams_raw_timings.csv \
    --output  results/b03_cuda_streams_benchmark.csv \
    --warmup-frames 20

## 3C. Precision Comparison Table (FP32 / FP16 / INT8)

In [ ]:
import pandas as pd

# Load benchmark summaries
def load_total(path, runtime):
    df = pd.read_csv(path)
    return df[df['stage'] == 'total'].assign(runtime=runtime)

rows = []
for rt, path in [
    ('trt-fp32',      'results/b03_trt_fp32_benchmark.csv'),
    ('trt-fp16',      'results/b03_trt_fp16_benchmark.csv'),
    ('trt-int8',      'results/b03_trt_int8_benchmark.csv'),
    ('cuda-streams',  'results/b03_cuda_streams_benchmark.csv'),
]:
    # Use B02 FP32/FP16 results if B03-specific ones aren't regenerated
    fallback = path.replace('b03_trt_fp32', 'b02_trt_fp32').replace('b03_trt_fp16', 'b02_trt_fp16')
    actual   = path if os.path.exists(path) else fallback
    if os.path.exists(actual):
        rows.append(load_total(actual, rt))

if rows:
    summary = pd.concat(rows, ignore_index=True)
    print(summary[['runtime', 'mean_ms', 'p50_ms', 'p99_ms']].to_string(index=False))

# Model file sizes
print('\nEngine sizes:')
for fname, label in [
    ('models/yolo11n_fp32.engine', 'FP32'),
    ('models/yolo11n_fp16.engine', 'FP16'),
    ('models/yolo11n_int8.engine', 'INT8'),
]:
    if os.path.exists(fname):
        mb = os.path.getsize(fname) / 1e6
        print(f'  {label}: {mb:.1f} MB')

In [ ]:
# GPU memory usage per precision
import subprocess

def measure_gpu_mem(engine_path, input_size=640):
    """Run one inference pass and report peak GPU memory."""
    import torch
    from src.trt_runner import TRTRunner
    import numpy as np

    torch.cuda.reset_peak_memory_stats()
    runner = TRTRunner(engine_path)
    dummy  = np.zeros((1, 3, input_size, input_size), dtype=np.float32)
    for _ in range(5):
        runner.infer(dummy)
    torch.cuda.synchronize()
    peak_mb = torch.cuda.max_memory_allocated() / 1e6
    del runner
    torch.cuda.empty_cache()
    return peak_mb

import sys
sys.path.insert(0, '.')

print('Peak GPU memory per precision:')
for path, label in [
    ('models/yolo11n_fp32.engine', 'FP32'),
    ('models/yolo11n_fp16.engine', 'FP16'),
    ('models/yolo11n_int8.engine', 'INT8'),
]:
    if os.path.exists(path):
        mem = measure_gpu_mem(path)
        print(f'  {label}: {mem:.1f} MB')

## Generate Batch 03 Plots

In [ ]:
import glob

b03_benchmarks = ' '.join(glob.glob('results/b03_*_benchmark.csv'))

!python3 scripts/plot_results.py \
    --benchmarks {b03_benchmarks} \
    --output-dir results/ \
    --prefix     b03_

## Download Results

In [ ]:
import shutil, glob

# Save all B03 results to Drive first
if USE_DRIVE:
    for f in glob.glob('results/b03_*'):
        dst = f'{DRIVE_ROOT}/results/{os.path.basename(f)}'
        shutil.copy(f, dst)
    print('B03 results saved to Drive.')

# Zip and download locally
!zip -q b03_results.zip results/b03_* models/yolo11n_int8.engine models/calibration.cache

from google.colab import files
files.download('b03_results.zip')
print('Download started: b03_results.zip')